In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

def extract_specific_timepoints(input_file, output_file, start_time=2336, end_time=2372, step=2):
    """
    Extract data at specific timepoints and save to a new CSV file.
    
    Parameters:
    -----------
    input_file : str
        Path to the input CSV file
    output_file : str
        Path to save the output CSV file
    start_time : float
        Starting time point (default: 2336)
    end_time : float
        Ending time point (default: 2372)
    step : float
        Step size between time points (default: 2)
    """
    try:
        # Read the CSV file
        df = pd.read_csv(input_file)
        
        # Create a list of target timepoints
        target_times = np.arange(start_time, end_time + step, step)
        
        # Check if 'Time' column exists
        if 'Time' not in df.columns:
            print(f"Warning: 'Time' column not found in {input_file}. Available columns: {df.columns}")
            return None
        
        # Function to find the closest time in the dataframe to a target time
        def find_closest_time(df, target_time):
            return df.iloc[(df['Time'] - target_time).abs().argsort()[0]]
        
        # Extract rows for each target time
        extracted_rows = []
        for target_time in target_times:
            closest_row = find_closest_time(df, target_time)
            extracted_rows.append(closest_row)
        
        # Create a new dataframe with the extracted rows
        extracted_df = pd.DataFrame(extracted_rows)
        
        # Save to a new CSV file
        extracted_df.to_csv(output_file, index=False)
        
        return extracted_df
    
    except Exception as e:
        print(f"Error processing {input_file}: {str(e)}")
        return None

def process_csv_files(input_folder, output_folder, start_time=2336, end_time=2372, step=2):
    """
    Process all CSV files in the input folder and save results to the output folder.
    
    Parameters:
    -----------
    input_folder : str
        Path to the folder containing CSV files
    output_folder : str
        Path to the folder where processed files will be saved
    start_time : float
        Starting time point
    end_time : float
        Ending time point
    step : float
        Step size between time points
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all CSV files in the input folder
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    if not csv_files:
        print(f"No CSV files found in {input_folder}")
        return
    
    # Process each CSV file
    processed_files = []
    for input_file in csv_files:
        # Get the filename without path
        filename = os.path.basename(input_file)
        
        # Create output filename
        output_file = os.path.join(output_folder, filename)
        
        print(f"Processing {filename}...")
        extracted_df = extract_specific_timepoints(input_file, output_file, start_time, end_time, step)
        
        if extracted_df is not None:
            processed_files.append((filename, extracted_df))
            print(f"Successfully processed and saved to {output_file}")
    
    return processed_files

def create_comparison_plots(processed_files, output_folder):
    """
    Create comparison plots for pairs of processed files if possible.
    
    Parameters:
    -----------
    processed_files : list
        List of tuples (filename, dataframe) of processed files
    output_folder : str
        Path to the folder where plots will be saved
    """
    # Group files by potential model/file types
    # This is a simple approach - group files that have similar names
    file_groups = {}
    
    for filename, df in processed_files:
        # Try to extract a base name (remove model/version indicators)
        base_name = filename.replace('model1', '').replace('model2', '')
        base_name = base_name.replace('_h1', '').replace('_h2', '')
        base_name = base_name.replace('_extracted_timepoints', '')
        
        if base_name not in file_groups:
            file_groups[base_name] = []
        
        file_groups[base_name].append((filename, df))
    
    # Create plots for file pairs
    for base_name, files in file_groups.items():
        if len(files) == 2:
            filename1, df1 = files[0]
            filename2, df2 = files[1]
            
            # Create plot output name
            plot_name = f"comparison_{base_name.replace('.csv', '')}.png"
            plot_path = os.path.join(output_folder, plot_name)
            
            # Check if both dataframes have necessary columns
            required_cols = ['Time', 'Compartment_1', 'Compartment_2', 'Compartment_3']
            if all(col in df1.columns for col in required_cols) and all(col in df2.columns for col in required_cols):
                plot_extracted_data(df1, df2, 
                                   filename1.replace('.csv', ''), 
                                   filename2.replace('.csv', ''),
                                   plot_path)
                print(f"Created comparison plot: {plot_path}")

def plot_extracted_data(df1, df2, model1_name="Model 1", model2_name="Model 2", output_file="comparison.png"):
    """
    Plot the extracted data for visual verification.
    
    Parameters:
    -----------
    df1 : pandas.DataFrame
        Dataframe containing the extracted data for model 1
    df2 : pandas.DataFrame
        Dataframe containing the extracted data for model 2
    model1_name : str
        Name to display for model 1 in the legend
    model2_name : str
        Name to display for model 2 in the legend
    output_file : str
        Path to save the plot
    """
    fig, axs = plt.subplots(3, 1, figsize=(10, 12))
    
    # Set shared x-label
    fig.text(0.5, 0.04, 'Time (hr)', ha='center', fontsize=12, fontweight='bold')
    
    # Plot for compartment 1
    axs[0].plot(df1['Time'], df1['Compartment_1'], 'bo-', label=f'{model1_name}')
    axs[0].plot(df2['Time'], df2['Compartment_1'], 'ro-', label=f'{model2_name}')
    axs[0].set_title('Compartment 1')
    axs[0].set_ylabel('Concentration')
    axs[0].legend()
    axs[0].grid(True)
    
    # Plot for compartment 2
    axs[1].plot(df1['Time'], df1['Compartment_2'], 'bo-', label=f'{model1_name}')
    axs[1].plot(df2['Time'], df2['Compartment_2'], 'ro-', label=f'{model2_name}')
    axs[1].set_title('Compartment 2')
    axs[1].set_ylabel('Concentration')
    axs[1].legend()
    axs[1].grid(True)
    
    # Plot for compartment 3
    axs[2].plot(df1['Time'], df1['Compartment_3'], 'bo-', label=f'{model1_name}')
    axs[2].plot(df2['Time'], df2['Compartment_3'], 'ro-', label=f'{model2_name}')
    axs[2].set_title('Compartment 3')
    axs[2].set_ylabel('Concentration')
    axs[2].legend()
    axs[2].grid(True)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    plt.close()


def main():
    # Define input and output folders
    input_folder = 'sleep_hypothesis/sim_files/global'  # Current directory
    output_folder = 'sleep_hypothesis/sim_files/global_preprocess'  # Preprocess folder
    
    # Define time parameters
    start_time = 2336
    end_time = 2372
    step = 2
            
    # Process all CSV files
    print(f"\nProcessing CSV files from {input_folder} to extract timepoints {start_time} to {end_time} with step {step}...")
    processed_files = process_csv_files(input_folder, output_folder, start_time, end_time, step)
    
    # Create comparison plots if possible
    if processed_files:
        create_comparison_plots(processed_files, output_folder)
        print(f"\nAll processing complete. Files saved to {output_folder}/")
    else:
        print("\nNo files were successfully processed.")

if __name__ == "__main__":
    main()


Processing CSV files from sleep_hypothesis/sim_files/global to extract timepoints 2336 to 2372 with step 2...
Processing global_model2_sA.csv...
Successfully processed and saved to sleep_hypothesis/sim_files/global_preprocess/global_model2_sA.csv
Processing global_model2_sbp.csv...
Successfully processed and saved to sleep_hypothesis/sim_files/global_preprocess/global_model2_sbp.csv
Processing global_model2_sbc_scp_sA.csv...
Successfully processed and saved to sleep_hypothesis/sim_files/global_preprocess/global_model2_sbc_scp_sA.csv
Processing global_model2_scp_sp.csv...
Successfully processed and saved to sleep_hypothesis/sim_files/global_preprocess/global_model2_scp_sp.csv
Processing global_model2_sA_sp.csv...
Successfully processed and saved to sleep_hypothesis/sim_files/global_preprocess/global_model2_sA_sp.csv
Processing global_model2_all_sigma.csv...
Successfully processed and saved to sleep_hypothesis/sim_files/global_preprocess/global_model2_all_sigma.csv
Processing global_mod

In [45]:
import numpy as np
import pandas as pd

# Load experimental data
name = "blattner"
exp = pd.read_csv(f"sleep_hypothesis/sim_files/{name}_csf_concentration.csv")
y_obs = exp["Concentration"].values

# List of model prediction files and their respective number of parameters
model_files1 = [
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbc.csv", 1),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_scp.csv", 1),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sA.csv", 1),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbp.csv", 1),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sp.csv", 1),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sA_sp.csv", 2),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbc_scp.csv", 2),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbp_sA.csv", 2),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbp_sp.csv", 2),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_scp_sp.csv", 2),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbc_sbp_scp.csv", 3),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbc_scp_sA.csv", 3),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbc_scp_sp.csv", 3),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbp_sA_sp.csv", 3),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbc_scp_sA_sp.csv", 4),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbc_scp_sbp_sA.csv", 4),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_sbc_scp_sbp_sp.csv", 4),
    (f"sleep_hypothesis/sim_files/{name}_preprocess/{name}_model2_all_sigma.csv", 5),
]

model_files2 = [
    (f"pressure_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc.csv", 1),
    (f"pressure_hypothesis/sim_files/{name}_preprocess/{name}_model2_rcp.csv", 1),
    (f"pressure_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc_rcp.csv", 2),
    (f"pressure_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbp_rp.csv", 2),
    (f"pressure_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc_rcp_rbp.csv", 3),
    (f"pressure_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc_rcp_rp.csv", 3),
]

model_files3 = [
    (f"combined_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc_sbc_scp.csv", 3),
    (f"combined_hypothesis/sim_files/{name}_preprocess/{name}_model2_rcp_sbc_scp.csv", 3),
    (f"combined_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc_scp_sp.csv", 3),
    (f"combined_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc_rcp_sbc_scp.csv", 4),
    (f"combined_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc_sbc_scp_sp.csv", 4),
    (f"combined_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc_rcp_rbp_sbc_scp_sbp.csv", 6),
    (f"combined_hypothesis/sim_files/{name}_preprocess/{name}_model2_rbc_rcp_all_sigma.csv", 7),
]

# Function to calculate AIC, AICc, and BIC
def calculate_aic_aicc_bic(y_obs, y_pred, num_params):
    residuals = y_obs - y_pred
    sd_residuals = np.std(residuals, ddof=0)
    n = len(y_obs)

    # Calculate residual sum of squares
    rss = np.sum(residuals**2)

    k = num_params + 1 # +1 for the variance parameter
    
    # MLE estimate of variance
    sigma2_mle = rss / n
    
    # Calculate log-likelihood for normal distribution
    log_likelihood = -n/2 * (np.log(2 * np.pi) + np.log(sigma2_mle) + 1)
    
    # Calculate log-likelihood
    #log_likelihood = -n / 2 * np.log(2 * np.pi) - n * np.log(sd_residuals) - np.sum(residuals**2) / (2 * sd_residuals**2)
    
    # Calculate AIC
    aic = 2 * k - 2 * log_likelihood
    
    # Calculate AICc (corrected AIC for small sample sizes)
    if n - k - 1 > 0:  # Check to avoid division by zero
        aicc = aic + (2 * k * (k + 1)) / (n - k - 1)
    else:
        aicc = np.inf  # Set to infinity if correction term is undefined
    
    # Calculate BIC
    bic = k * np.log(n) - 2 * log_likelihood
    
    return aic, aicc, bic, residuals, sd_residuals

# Calculate AIC, AICc, and BIC for each model
results1 = []
for file_path, num_params in model_files1:
    model_df = pd.read_csv(file_path)
    y_pred = model_df["Compartment_2"].values
    
    # Make sure experimental and predicted data have the same length
    if len(y_pred) != len(y_obs):
        print(f"Length mismatch for {file_path}: {len(y_pred)} != {len(y_obs)}")
        continue
    
    aic, aicc, bic, _, _ = calculate_aic_aicc_bic(y_obs, y_pred, num_params)
    results1.append((file_path, num_params, aic, aicc, bic))

results2 = []
for file_path, num_params in model_files2:
    model_df = pd.read_csv(file_path)
    y_pred = model_df["Compartment_2"].values
    
    # Make sure experimental and predicted data have the same length
    if len(y_pred) != len(y_obs):
        print(f"Length mismatch for {file_path}: {len(y_pred)} != {len(y_obs)}")
        continue
    
    aic, aicc, bic, _, _ = calculate_aic_aicc_bic(y_obs, y_pred, num_params)
    results2.append((file_path, num_params, aic, aicc, bic))

results3 = []
for file_path, num_params in model_files3:
    model_df = pd.read_csv(file_path)
    y_pred = model_df["Compartment_2"].values
    
    # Make sure experimental and predicted data have the same length
    if len(y_pred) != len(y_obs):
        print(f"Length mismatch for {file_path}: {len(y_pred)} != {len(y_obs)}")
        continue
    
    aic, aicc, bic, _, _ = calculate_aic_aicc_bic(y_obs, y_pred, num_params)
    results3.append((file_path, num_params, aic, aicc, bic))

# Display results
results_df1 = pd.DataFrame(results1, columns=["Model File", "Num Params", "AIC", "AICc", "BIC"])
results_df2 = pd.DataFrame(results2, columns=["Model File", "Num Params", "AIC", "AICc", "BIC"])
results_df3 = pd.DataFrame(results3, columns=["Model File", "Num Params", "AIC", "AICc", "BIC"])
print(results_df1)
print(results_df2)
print(results_df3)

# Optional: Show which models are best according to each criterion
print("\nBest models according to each criterion:")
best_aic_idx = results_df["AIC"].idxmin()
best_aicc_idx = results_df["AICc"].idxmin()
best_bic_idx = results_df["BIC"].idxmin()

print(f"Best AIC: {results_df.loc[best_aic_idx, 'Model File']} (AIC = {results_df.loc[best_aic_idx, 'AIC']:.2f})")
print(f"Best AICc: {results_df.loc[best_aicc_idx, 'Model File']} (AICc = {results_df.loc[best_aicc_idx, 'AICc']:.2f})")
print(f"Best BIC: {results_df.loc[best_bic_idx, 'Model File']} (BIC = {results_df.loc[best_bic_idx, 'BIC']:.2f})")

                                           Model File  Num Params         AIC  \
0   sleep_hypothesis/sim_files/blattner_preprocess...           1  238.451126   
1   sleep_hypothesis/sim_files/blattner_preprocess...           1  187.746181   
2   sleep_hypothesis/sim_files/blattner_preprocess...           1  233.034988   
3   sleep_hypothesis/sim_files/blattner_preprocess...           1  218.359953   
4   sleep_hypothesis/sim_files/blattner_preprocess...           1  238.432896   
5   sleep_hypothesis/sim_files/blattner_preprocess...           2  235.034988   
6   sleep_hypothesis/sim_files/blattner_preprocess...           2  187.744665   
7   sleep_hypothesis/sim_files/blattner_preprocess...           2  210.730303   
8   sleep_hypothesis/sim_files/blattner_preprocess...           2  220.359953   
9   sleep_hypothesis/sim_files/blattner_preprocess...           2  189.746181   
10  sleep_hypothesis/sim_files/blattner_preprocess...           3  189.746228   
11  sleep_hypothesis/sim_fil

In [51]:
import numpy as np
import pandas as pd

def heteroscedastic_aic(csf_obs, plasma_obs, csf_pred, plasma_pred, k):
    """
    Calculate heteroscedastic AIC/AICc/BIC with separate variance for CSF and Plasma.
    
    Parameters:
        csf_obs, plasma_obs: observed values
        csf_pred, plasma_pred: predicted values
        k: number of model parameters (NOT including variance parameters)
    
    Returns:
        dict with all metrics
    """
    # CSF compartment
    csf_resid = csf_obs - csf_pred
    n_csf = len(csf_obs)
    rss_csf = np.sum(csf_resid**2)
    sigma2_csf = rss_csf / n_csf
    logL_csf = -0.5 * n_csf * (np.log(2 * np.pi * sigma2_csf) + 1)
    
    # Plasma compartment
    plasma_resid = plasma_obs - plasma_pred
    n_plasma = len(plasma_obs)
    rss_plasma = np.sum(plasma_resid**2)
    sigma2_plasma = rss_plasma / n_plasma
    logL_plasma = -0.5 * n_plasma * (np.log(2 * np.pi * sigma2_plasma) + 1)
    
    # Total
    n_total = n_csf + n_plasma
    logL_total = logL_csf + logL_plasma

    k_new = k + 2
    
    # Information criteria (k does NOT include variance parameters in heteroscedastic models)
    aic = 2 * k_new - 2 * logL_total
    
    if n_total - k_new - 1 > 0:
        aicc = aic + (2 * k_new * (k_new + 1)) / (n_total - k_new - 1)
    else:
        aicc = np.inf
    
    bic = k_new * np.log(n_total) - 2 * logL_total
    
    return {
        'csf': {
            'n': n_csf,
            'rss': rss_csf,
            'sigma': np.sqrt(sigma2_csf),
            'logL': logL_csf
        },
        'plasma': {
            'n': n_plasma,
            'rss': rss_plasma,
            'sigma': np.sqrt(sigma2_plasma),
            'logL': logL_plasma
        },
        'total': {
            'n': n_total,
            'rss': rss_csf + rss_plasma,
            'logL': logL_total,
            'aic': aic,
            'aicc': aicc,
            'aicc/n': aicc / n_total,
            'bic': bic,
        }
    }


def load_and_prepare_data(csf_exp_file, plasma_exp_file, model_file):
    """
    Load experimental and model data from CSV files.
    Round experimental data to 3 decimal places.
    
    Parameters:
        csf_exp_file: path to CSF experimental data CSV
        plasma_exp_file: path to Plasma experimental data CSV
        model_file: path to model prediction CSV
    
    Returns:
        csf_obs, plasma_obs, csf_pred, plasma_pred as numpy arrays
    """
    csf_exp_df = pd.read_csv(csf_exp_file)
    plasma_exp_df = pd.read_csv(plasma_exp_file)
    model_df = pd.read_csv(model_file)
    
    csf_obs = np.round(csf_exp_df['Concentration'].values, 3)
    plasma_obs = np.round(plasma_exp_df['Concentration'].values, 3)
    
    csf_pred = model_df['Compartment_2'].values
    plasma_pred = model_df['Compartment_3'].values
    
    if len(csf_obs) != len(csf_pred):
        raise ValueError(f"CSF length mismatch: exp={len(csf_obs)}, model={len(csf_pred)}")
    if len(plasma_obs) != len(plasma_pred):
        raise ValueError(f"Plasma length mismatch: exp={len(plasma_obs)}, model={len(plasma_pred)}")
    
    return csf_obs, plasma_obs, csf_pred, plasma_pred


def analyze_model(csf_exp_file, plasma_exp_file, model_file, k, model_name):
    """
    Analyze a single model using the heteroscedastic approach.
    """
    csf_obs, plasma_obs, csf_pred, plasma_pred = load_and_prepare_data(
        csf_exp_file, plasma_exp_file, model_file
    )
    
    print("=" * 100)
    print(f"{model_name} ANALYSIS (k={k} parameters)")
    print("=" * 100)
    
    hetero = heteroscedastic_aic(csf_obs, plasma_obs, csf_pred, plasma_pred, k)
    
    print("\nHETEROSCEDASTIC APPROACH (separate variance for CSF and Plasma):")
    print("-" * 100)
    print(f"{'Compartment':<15} {'n':>6} {'RSS':>12} {'Sigma':>10} {'LogL':>12}")
    print("-" * 100)
    print(f"{'CSF':<15} {hetero['csf']['n']:>6} {hetero['csf']['rss']:>12.2f} "
          f"{hetero['csf']['sigma']:>10.4f} "
          f"{hetero['csf']['logL']:>12.4f}")
    print(f"{'Plasma':<15} {hetero['plasma']['n']:>6} {hetero['plasma']['rss']:>12.2f} "
          f"{hetero['plasma']['sigma']:>10.4f} "
          f"{hetero['plasma']['logL']:>12.4f}")
    print("-" * 100)
    print(f"{'TOTAL':<15} {hetero['total']['n']:>6} {hetero['total']['rss']:>12.2f}")
    print(f"Total Log-likelihood: {hetero['total']['logL']:.4f}")
    print(f"\nAIC:    {hetero['total']['aic']:.4f}")
    print(f"AICc:   {hetero['total']['aicc']:.4f}")
    print(f"AICc/n: {hetero['total']['aicc/n']:.4f}")
    print(f"BIC:    {hetero['total']['bic']:.4f}")
    
    return hetero


def compare_models(models_config):
    """
    Compare multiple models.
    
    Parameters:
        models_config: list of dicts with keys:
            - 'name': model name
            - 'csf_exp': path to CSF experimental data
            - 'plasma_exp': path to Plasma experimental data
            - 'model': path to model predictions
            - 'k': number of parameters
    """
    hetero_results = []
    
    for config in models_config:
        hetero = analyze_model(
            config['csf_exp'],
            config['plasma_exp'],
            config['model'],
            config['k'],
            config['name']
        )
        
        hetero_results.append({
            'name': config['name'],
            'k': config['k'],
            'aic': hetero['total']['aic'],
            'aicc': hetero['total']['aicc'],
            'aicc/n': hetero['total']['aicc/n'],
            'bic': hetero['total']['bic'],
        })
    
    print("\n" + "=" * 100)
    print("FINAL COMPARISON - HETEROSCEDASTIC APPROACH")
    print("=" * 100)
    print(f"{'Model':<30} {'k':>4} {'AIC':>12} {'AICc':>12} {'AICc/n':>12} {'BIC':>12}")
    print("-" * 100)
    
    for r in hetero_results:
        print(f"{r['name']:<30} {r['k']:>4} {r['aic']:>12.4f} {r['aicc']:>12.4f} "
              f"{r['aicc/n']:>12.4f} {r['bic']:>12.4f}")
    
    if len(hetero_results) == 2:
        print("-" * 100)
        delta_aic  = hetero_results[1]['aic']  - hetero_results[0]['aic']
        delta_aicc = hetero_results[1]['aicc'] - hetero_results[0]['aicc']
        delta_bic  = hetero_results[1]['bic']  - hetero_results[0]['bic']
        
        print(f"{'Delta (Model2 - Model1)':<30} {' ':>4} {delta_aic:>12.4f} {delta_aicc:>12.4f} "
              f"{delta_bic:>12.4f}")
        
        print("\n" + "-" * 100)
        if hetero_results[0]['aicc'] < hetero_results[1]['aicc']:
            print(f"→ {hetero_results[0]['name']} is BETTER (ΔAICc = {delta_aicc:.4f})")
        else:
            print(f"→ {hetero_results[1]['name']} is BETTER (ΔAICc = {-delta_aicc:.4f})")
    
    print("-" * 100)


# ===== MAIN EXECUTION =====
if __name__ == "__main__":

    BASE_DIR   = 'combined_hypothesis/sim_files'
    CSF_EXP    = f'{BASE_DIR}/liu_csf_concentration.csv'
    PLASMA_EXP = f'{BASE_DIR}/liu_plasma_concentration.csv'

    # Override k for folders where token count doesn't reflect true parameter count
    K_OVERRIDES = {
        "rbc_rcp_all_sigma": 7,
    }

    def infer_k(folder_name: str) -> int:
        """Infer k from token count, with manual overrides for special cases."""
        return K_OVERRIDES.get(folder_name, len(folder_name.split('_')))

    folders = [
        "rbc_rcp_all_sigma",
        "rbc_rcp_rbp_sbc_scp_sbp",
        "rbc_rcp_sbc_scp",
        "rbc_sbc_scp",
        "rbc_sbc_scp_sp",
        "rbc_scp_sp",
        "rcp_sbc_scp"
    ]

    models_config = [
        {
            'name':      f'Model 1 ({folder})',
            'csf_exp':   CSF_EXP,
            'plasma_exp': PLASMA_EXP,
            'model':     f'{BASE_DIR}/liu_preprocess/liu_model2_{folder}.csv',
            'k':         infer_k(folder),
        }
        for folder in folders
    ]

    compare_models(models_config)

Model 1 (rbc_rcp_all_sigma) ANALYSIS (k=7 parameters)

HETEROSCEDASTIC APPROACH (separate variance for CSF and Plasma):
----------------------------------------------------------------------------------------------------
Compartment          n          RSS      Sigma         LogL
----------------------------------------------------------------------------------------------------
CSF                 19     43210.35    47.6889    -100.3891
Plasma              19         3.60     0.4354     -11.1609
----------------------------------------------------------------------------------------------------
TOTAL               38     43213.95
Total Log-likelihood: -111.5500

AIC:    241.1000
AICc:   247.5286
AICc/n: 6.5139
BIC:    255.8383
Model 1 (rbc_rcp_rbp_sbc_scp_sbp) ANALYSIS (k=6 parameters)

HETEROSCEDASTIC APPROACH (separate variance for CSF and Plasma):
----------------------------------------------------------------------------------------------------
Compartment          n          RSS

In [57]:
import numpy as np
import pandas as pd

def heteroscedastic_aic(csf_obs1, csf_obs2, csf_obs3, plasma_obs, csf_pred, plasma_pred, k):
    """
    Calculate heteroscedastic AIC/AICc/BIC with separate variance for CSF and Plasma.
    
    Parameters:
        csf_obs, plasma_obs: observed values
        csf_pred, plasma_pred: predicted values
        k: number of model parameters (NOT including variance parameters)
    
    Returns:
        dict with all metrics
    """
    # CSF compartment1
    csf_resid1 = csf_obs1 - csf_pred
    n_csf1 = len(csf_obs1)
    rss_csf1 = np.sum(csf_resid1**2)
    sigma2_csf1 = rss_csf1 / n_csf1
    logL_csf1 = -0.5 * n_csf1 * (np.log(2 * np.pi * sigma2_csf1) + 1)

    # CSF compartment2
    csf_resid2 = csf_obs2 - csf_pred
    n_csf2 = len(csf_obs2)
    rss_csf2 = np.sum(csf_resid2**2)
    sigma2_csf2 = rss_csf2 / n_csf2
    logL_csf2 = -0.5 * n_csf2 * (np.log(2 * np.pi * sigma2_csf2) + 1)

    # CSF compartment3
    csf_resid3 = csf_obs3 - csf_pred
    n_csf3 = len(csf_obs3)
    rss_csf3 = np.sum(csf_resid3**2)
    sigma2_csf3 = rss_csf3 / n_csf3
    logL_csf3 = -0.5 * n_csf3 * (np.log(2 * np.pi * sigma2_csf3) + 1)
    
    # Plasma compartment
    plasma_resid = plasma_obs - plasma_pred
    n_plasma = len(plasma_obs)
    rss_plasma = np.sum(plasma_resid**2)
    sigma2_plasma = rss_plasma / n_plasma
    logL_plasma = -0.5 * n_plasma * (np.log(2 * np.pi * sigma2_plasma) + 1)
    
    # Total
    n_total = n_csf1 + n_csf2 + n_csf3 + n_plasma
    logL_total = logL_csf1 + logL_csf2 + logL_csf3 + logL_plasma

    k_new = k + 4
    
    # Information criteria (k does NOT include variance parameters in heteroscedastic models)
    aic = 2 * k_new - 2 * logL_total
    
    if n_total - k_new - 1 > 0:
        aicc = aic + (2 * k_new * (k_new + 1)) / (n_total - k_new - 1)
    else:
        aicc = np.inf
    
    bic = k_new * np.log(n_total) - 2 * logL_total
    
    return {
        'csf1': {
            'n': n_csf1,
            'rss': rss_csf1,
            'sigma': np.sqrt(sigma2_csf1),
            'logL': logL_csf1
        },
        'csf2': {
            'n': n_csf2,
            'rss': rss_csf2,
            'sigma': np.sqrt(sigma2_csf2),
            'logL': logL_csf2
        },
        'csf3': {
            'n': n_csf3,
            'rss': rss_csf3,
            'sigma': np.sqrt(sigma2_csf3),
            'logL': logL_csf3
        },
        'plasma': {
            'n': n_plasma,
            'rss': rss_plasma,
            'sigma': np.sqrt(sigma2_plasma),
            'logL': logL_plasma
        },
        'total': {
            'n': n_total,
            'rss': rss_csf1 + rss_csf2 + rss_csf3 + rss_plasma,
            'logL': logL_total,
            'aic': aic,
            'aicc': aicc,
            'aicc/n': aicc/n_total,
            'bic': bic,
        }
    }

def load_and_prepare_data(csf_exp_file1, csf_exp_file2, csf_exp_file3, plasma_exp_file, model_file):
    """
    Load experimental and model data from CSV files.
    Round experimental data to 3 decimal places.
    
    Parameters:
        csf_exp_file: path to CSF experimental data CSV
        plasma_exp_file: path to Plasma experimental data CSV
        model_file: path to model prediction CSV
    
    Returns:
        csf_obs, plasma_obs, csf_pred, plasma_pred as numpy arrays
    """
    # Load experimental data
    csf_exp_df1 = pd.read_csv(csf_exp_file1)
    csf_exp_df2 = pd.read_csv(csf_exp_file2)
    csf_exp_df3 = pd.read_csv(csf_exp_file3)
    plasma_exp_df = pd.read_csv(plasma_exp_file)
    
    # Load model data
    model_df = pd.read_csv(model_file)
    
    # Round experimental data to 3 decimal places
    csf_obs1 = np.round(csf_exp_df1['Concentration'].values, 3)
    csf_obs2 = np.round(csf_exp_df2['Concentration'].values, 3)
    csf_obs3 = np.round(csf_exp_df3['Concentration'].values, 3)
    plasma_obs = np.round(plasma_exp_df['Concentration'].values, 3)
    
    # Extract model predictions (already should be 3 decimal places if saved correctly)
    csf_pred = model_df['Compartment_2'].values
    plasma_pred = model_df['Compartment_3'].values
    
    # Verify lengths match
    if len(csf_obs1) != len(csf_pred):
        raise ValueError(f"CSF length mismatch: exp={len(csf_obs)}, model={len(csf_pred)}")
    if len(plasma_obs) != len(plasma_pred):
        raise ValueError(f"Plasma length mismatch: exp={len(plasma_obs)}, model={len(plasma_pred)}")
    
    return csf_obs1, csf_obs2, csf_obs3, plasma_obs, csf_pred, plasma_pred

def analyze_model(csf_exp_file1, csf_exp_file2, csf_exp_file3, plasma_exp_file, model_file, k, model_name):
    """
    Analyze a single model using both heteroscedastic and homoscedastic approaches.
    """
    # Load data
    csf_obs1, csf_obs2, csf_obs3, plasma_obs, csf_pred, plasma_pred = load_and_prepare_data(
        csf_exp_file1, csf_exp_file2, csf_exp_file3, plasma_exp_file, model_file
    )
    
    print("="*100)
    print(f"{model_name} ANALYSIS (k={k} parameters)")
    print("="*100)
    
    # Heteroscedastic approach
    hetero = heteroscedastic_aic(csf_obs1, csf_obs2, csf_obs3, plasma_obs, csf_pred, plasma_pred, k)
    
    print("\nHETEROSCEDASTIC APPROACH (separate variance for CSF and Plasma):")
    print("-"*100)
    print(f"{'Compartment':<15} {'n':>6} {'RSS':>12} {'Sigma':>10} {'RMSE':>10} {'NRMSE':>10} {'LogL':>12}")
    print("-"*100)
    print(f"{'CSF1':<15} {hetero['csf1']['n']:>6} {hetero['csf1']['rss']:>12.2f} "
          f"{hetero['csf1']['sigma']:>10.4f} "
          f"{hetero['csf1']['logL']:>12.4f}")
    print(f"{'CSF2':<15} {hetero['csf2']['n']:>6} {hetero['csf2']['rss']:>12.2f} "
          f"{hetero['csf2']['sigma']:>10.4f} "
          f"{hetero['csf2']['logL']:>12.4f}")
    print(f"{'CSF3':<15} {hetero['csf3']['n']:>6} {hetero['csf3']['rss']:>12.2f} "
          f"{hetero['csf3']['sigma']:>10.4f} "
          f"{hetero['csf3']['logL']:>12.4f}")
    print(f"{'Plasma':<15} {hetero['plasma']['n']:>6} {hetero['plasma']['rss']:>12.2f} "
          f"{hetero['plasma']['sigma']:>10.4f} "
          f"{hetero['plasma']['logL']:>12.4f}")
    print("-"*100)
    print(f"{'TOTAL':<15} {hetero['total']['n']:>6} {hetero['total']['rss']:>12.2f}")
    print(f"Total Log-likelihood: {hetero['total']['logL']:.4f}")
    print(f"\nAIC:  {hetero['total']['aic']:.4f}")
    print(f"AICc: {hetero['total']['aicc']:.4f}")
    print(f"AICc/n: {hetero['total']['aicc/n']:.4f}")
    print(f"BIC:  {hetero['total']['bic']:.4f}")
    
    return hetero

def compare_models(models_config):
    """
    Compare multiple models.
    
    Parameters:
        models_config: list of dicts with keys:
            - 'name': model name
            - 'csf_exp': path to CSF experimental data
            - 'plasma_exp': path to Plasma experimental data
            - 'model': path to model predictions
            - 'k': number of parameters
    """
    hetero_results = []
    
    # Analyze each model
    for config in models_config:
        hetero = analyze_model(
            config['csf_exp1'],
            config['csf_exp2'],
            config['csf_exp3'],
            config['plasma_exp'],
            config['model'],
            config['k'],
            config['name']
        )
        
        hetero_results.append({
            'name': config['name'],
            'k': config['k'],
            'aic': hetero['total']['aic'],
            'aicc': hetero['total']['aicc'],
            'aicc/n': hetero['total']['aicc/n'],
            'bic': hetero['total']['bic'],
        })
    
    # Print comparison
    print("\n" + "="*100)
    print("FINAL COMPARISON - HETEROSCEDASTIC APPROACH")
    print("="*100)
    print(f"{'Model':<30} {'k':>4} {'AIC':>12} {'AICc':>12} {'AICc/n':>12} {'BIC':>12}")
    print("-"*100)
    
    for r in hetero_results:
        print(f"{r['name']:<30} {r['k']:>4} {r['aic']:>12.4f} {r['aicc']:>12.4f} "
              f"{r['aicc/n']:>12.4f} {r['bic']:>12.4f}")
    
    # Calculate deltas if comparing exactly 2 models
    if len(hetero_results) == 2:
        print("-"*100)
        delta_aic = hetero_results[1]['aic'] - hetero_results[0]['aic']
        delta_aicc = hetero_results[1]['aicc'] - hetero_results[0]['aicc']
        delta_bic = hetero_results[1]['bic'] - hetero_results[0]['bic']
        
        print(f"{'Delta (Model2 - Model1)':<30} {' ':>4} {delta_aic:>12.4f} {delta_aicc:>12.4f} "
              f"{delta_bic:>12.4f}")
        
        print("\n" + "-"*100)
        if hetero_results[0]['aicc'] < hetero_results[1]['aicc']:
            print(f"→ {hetero_results[0]['name']} is BETTER (ΔAICc = {delta_aicc:.4f})")
        else:
            print(f"→ {hetero_results[1]['name']} is BETTER (ΔAICc = {-delta_aicc:.4f})")
    
    print("-"*100)

# ===== MAIN EXECUTION =====
if __name__ == "__main__":

    BASE_DIR   = 'combined_hypothesis/sim_files'
    CSF_EXP1    = f'{BASE_DIR}/blattner_csf_concentration.csv'
    CSF_EXP2    = f'{BASE_DIR}/lucey_csf_concentration.csv'
    CSF_EXP3    = f'{BASE_DIR}/liu_csf_concentration.csv'
    PLASMA_EXP = f'{BASE_DIR}/liu_plasma_concentration.csv'

    # Override k for folders where token count doesn't reflect true parameter count
    K_OVERRIDES = {
        "rbc_rcp_all_sigma": 7,
    }

    def infer_k(folder_name: str) -> int:
        """Infer k from token count, with manual overrides for special cases."""
        return K_OVERRIDES.get(folder_name, len(folder_name.split('_')))

    folders = [
        "rbc_rcp_all_sigma",
        "rbc_rcp_rbp_sbc_scp_sbp",
        "rbc_rcp_sbc_scp",
        "rbc_sbc_scp",
        "rbc_sbc_scp_sp",
        "rbc_scp_sp",
        "rcp_sbc_scp"
    ]

    models_config = [
        {
            'name':      f'Model 1 ({folder})',
            'csf_exp1':   CSF_EXP1,
            'csf_exp2':   CSF_EXP2,
            'csf_exp3':   CSF_EXP3,
            'plasma_exp': PLASMA_EXP,
            'model':     f'{BASE_DIR}/liu_preprocess/liu_model2_{folder}.csv',
            'k':         infer_k(folder),
        }
        for folder in folders
    ]

    compare_models(models_config)

Model 1 (rbc_rcp_all_sigma) ANALYSIS (k=7 parameters)

HETEROSCEDASTIC APPROACH (separate variance for CSF and Plasma):
----------------------------------------------------------------------------------------------------
Compartment          n          RSS      Sigma       RMSE      NRMSE         LogL
----------------------------------------------------------------------------------------------------
CSF1                19    102932.88    73.6038    -108.6351
CSF2                19     60296.70    56.3339    -103.5545
CSF3                19     43210.35    47.6889    -100.3891
Plasma              19         3.60     0.4354     -11.1609
----------------------------------------------------------------------------------------------------
TOTAL               76    206443.53
Total Log-likelihood: -323.7396

AIC:  669.4791
AICc: 673.6041
AICc/n: 8.8632
BIC:  695.1172
Model 1 (rbc_rcp_rbp_sbc_scp_sbp) ANALYSIS (k=6 parameters)

HETEROSCEDASTIC APPROACH (separate variance for CSF and Plasma):
